# ESP-LLM — Train ESP32-S3 model on Colab T4

Clones this repo and trains the **ESP32-S3 (~51 M-param, 12-layer MoE)** model with one command:
`python main.py --target=esp32s3 --train`

> **Setup:** in Colab go to `Runtime > Change runtime type > T4 GPU` (Python 3), then `Runtime > Run all`.

## ⏱️ How long on a T4?

| Run | Iters × batch × ctx | Tokens | Expected wall time on T4 |
|---|---|---|---|
| **ESP32-S3 full** (`max_iters=25000`, `batch=32`, `ctx=192`) | 25,000 × 6,144 tok/iter | ~154 M | **~12–18 h** (typically **~8–14 h** with early stopping) |

Why:
- S3 config = 51.2 M total params, ~3.4 M active/token (top-1 MoE), ~150–160 GFLOP/iter fwd+bwd.
- The per-layer Python `for expert in 28 experts` loop is launch-overhead bound, so a T4 is only ~1.5–2× faster than Apple Silicon (README: ~1 day on Apple Silicon).
- Measured reference: single MoE block fwd+bwd ≈ 0.3 s on M2 → full 12-layer step ≈ 4–5 s on M2 → ≈ 2–3 s/step on T4 → 25,000 × ~2.5 s ≈ 17 h. Early stopping (`eval_interval=200`, `patience=12`) usually cuts this to 12k–20k iters.
- ⚠️ Free Colab T4 sessions cap at ~12 h and idle-disconnect: mount Drive (optional cell below) or use Colab Pro / L4 / A100 for a guaranteed single-session finish.

In [ ]:
# 1 — Verify T4 GPU
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
# 2 — Clone the repo (re-runnable)
!rm -rf /content/espllm
!git clone https://github.com/ahmedbarakat207/espllm.git /content/espllm
!ls -lh /content/espllm | head -30
!du -h /content/espllm/dataset.txt /content/espllm/bpe-vocab.json /content/espllm/bpe-merges.txt

In [ ]:
# 3 — Install deps (Colab already ships torch+CUDA; only add what's missing)
%cd /content/espllm
!pip install -q tokenizers torchao

In [ ]:
# 4 (optional) — Persist checkpoints to Drive so a Colab eviction doesn't lose them
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p "/content/drive/MyDrive/espllm_checkpoints" && echo ok

In [ ]:
# 5 — TRAIN (this is the only command you need)
%cd /content/espllm
!python main.py --target=esp32s3 --train

## After training

The run writes (in `/content/espllm/model/`):

- `model_esp32s3.pt` — full-precision weights
- `model_esp32s3.pt.best` — best early-stopped weights (lowest val loss)
- `model_esp32s3.pt.quantized` — post-training quantized artifact (this is what `convert_model_to_c.py` / `flash.py` consumes)

Verify + back up:

In [ ]:
# 6 — Verify outputs
%cd /content/espllm
!ls -lh model/model_esp32s3*

# Optional: copy to Drive
# !cp model/model_esp32s3.pt* "/content/drive/MyDrive/espllm_checkpoints/" && ls -lh "/content/drive/MyDrive/espllm_checkpoints/"

# Optional: download quantized file to your laptop (Colab file browser, or):
# from google.colab import files; files.download('model/model_esp32s3.pt.quantized')

## Notes

- `dataset.txt` (~5.6 MB, ~100k pairs) and the BPE files are already in the repo, so no rebuild is needed. To regenerate: `!python build_dataset.py`.
- Progress prints every 200 iters (`train/val loss | lr`), best checkpoint saves automatically, early stopping (`patience=12`) ends the run when val loss stalls.
- Other targets with the same notebook: `!python main.py --target=esp32 --train` (~10 M params, faster) or `!python main.py --target=esp8266 --train` (~1.8 M params, fastest).
- Next step locally: `python flash.py esp32s3` (re-exports `src/model_weights.hpp` and flashes the S3 N16R8 board).